# 使用 `agr` 部署 httpbin：最短完整链路

本 Notebook 创建一个自定义 Sandbox Tool 和一个 Deployment，分别通过本地调试代理与生产数据面域名访问 httpbin，最后删除所有资源。这里不展开弹性、生命周期和会话亲和；它们由相邻 Notebook 独立讲解。

> 每个命令都直接执行。资源 ID 不会自动提取，请从输出中复制并在下一单元设置。Notebook 不保存执行输出。

## 1. 检查 AGR 配置

确认当前凭证和 region 有效。后续所有命令都显式使用 `AGR_REGION`，避免 Notebook 与本机默认配置不一致。把 `AGR_ROLE_ARN` 替换为允许 AGR 拉取目标 CCR 镜像的 CAM 角色 ARN。

In [ ]:
%env AGR_REGION=ap-shanghai
%env AGR_DOMAIN=tencentags.com
%env AGR_ROLE_ARN=qcs::cam::uin/replace-me:roleName/replace-me
!agr status

## 2. 创建 httpbin Sandbox Tool

将 `your-name` 改为个人唯一后缀。Tool 使用固定版本公共镜像，并向 Deployment 暴露容器的 `8080` 端口。`--wait` 会等待 Tool 进入最终状态。

In [ ]:
%env HTTPBIN_TOOL_NAME=httpbin-simple-your-name
!agr tool create \
  --region "$AGR_REGION" \
  --tool-name "$HTTPBIN_TOOL_NAME" \
  --tool-type custom \
  --persistent \
  --role-arn "$AGR_ROLE_ARN" \
  --network-configuration '{"NetworkMode":"PUBLIC"}' \
  --custom-configuration '{"Image":"ccr.ccs.tencentyun.com/ags.dev/go-httpbin:v2.25.0","ImageRegistryType":"personal","Command":["/bin/go-httpbin"],"Args":["-host","0.0.0.0","-port","8080"],"Env":[{"Name":"EXCLUDE_HEADERS","Value":"X-Access-Token"}],"Ports":[{"Name":"http","Port":8080,"Protocol":"TCP"}],"Resources":{"CPU":"200m","Memory":"500Mi"},"Probe":{"HttpGet":{"Path":"/status/200","Port":8080,"Scheme":"HTTP"},"ReadyTimeoutMs":30000,"ProbeTimeoutMs":1000,"ProbePeriodMs":3000,"SuccessThreshold":1,"FailureThreshold":10}}' \
  --wait

## 3. 创建 Deployment

从上一单元输出复制 `ToolId`，替换下面的占位值。Deployment 名称必须是当前账号下唯一的 DNS-1123 名称。省略可选配置时，服务使用默认伸缩与生命周期设置。

In [ ]:
%env HTTPBIN_TOOL_ID=sdt-replace-me
%env HTTPBIN_DEPLOYMENT_NAME=httpbin-simple-your-name
!agr deployment create \
  --region "$AGR_REGION" \
  --deployment-name "$HTTPBIN_DEPLOYMENT_NAME" \
  --tool-id "$HTTPBIN_TOOL_ID"

## 4. 查询 Deployment

从创建输出复制 `DeploymentId`。`get` 查看一个 Deployment 的完整摘要，`list` 查看当前 region 中的 Deployment 列表。

In [ ]:
%env HTTPBIN_DEPLOYMENT_ID=dpl-replace-me
!agr deployment get "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr deployment list --region "$AGR_REGION"

## 5. 本地调试：Deployment proxy

`proxy` 是本地调试工具，不是生产接入方式。运行下一单元后，代理会占用该单元并监听 `127.0.0.1:18080`。请在另一个终端执行 `curl http://127.0.0.1:18080/get`；验证完成后回到 Notebook 中中断代理，再继续。

In [ ]:
!agr deployment proxy "$HTTPBIN_DEPLOYMENT_ID" 18080:8080 --region "$AGR_REGION"

## 6. 生产访问：短期 Token 与 Deployment 域名

生产客户端应获取短期 Deployment Token，并直接请求数据面域名。`AcquireDeploymentToken` 目前通过原始 API 命令调用；返回的 `Token` 只适用于目标 Deployment，并在 `ExpiresAt` 指定的时间过期。

Deployment HTTP 端口的域名规则是：

```text
https://{port}-{deployment-id}.{region}.agents.{data-plane-domain}
```

默认数据面域名为 `tencentags.com`，httpbin 端口为 `8080`。不要记录或提交真实 Token。

In [ ]:
!agr api call AcquireDeploymentToken \
  --region "$AGR_REGION" \
  --request '{"DeploymentId":"'$HTTPBIN_DEPLOYMENT_ID'"}' \
  --output json

从上一单元输出复制 `Data.Response.Response.Token`，设置后直接访问生产域名。请求通过 `X-Access-Token` header 携带 Token。Tool 已配置 httpbin 不回显该敏感 header。

In [ ]:
%env HTTPBIN_DEPLOYMENT_TOKEN=dpt-replace-me
!curl --fail-with-body --silent --show-error \
  --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" \
  "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"

## 7. 清理资源

先删除 Deployment 并等待异步删除完成，再删除 Tool。即使前面的访问步骤失败，也应执行本节。

In [ ]:
!agr deployment delete "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr instance list --tool-id "$HTTPBIN_TOOL_ID" --region "$AGR_REGION"

如果列表中仍有 `RUNNING`、`PAUSED` 或其他非 `STOPPED` 实例，复制其 ID，在新单元中执行以下命令；有多个实例时逐个重复。然后删除 Tool。

```text
%env HTTPBIN_INSTANCE_ID=replace-me
!agr instance delete "$HTTPBIN_INSTANCE_ID" --region "$AGR_REGION" --yes --wait
```

In [ ]:
!agr tool delete "$HTTPBIN_TOOL_ID" --region "$AGR_REGION" --yes --wait